# Assignment 1 — Google Colab CPU runner

This notebook clones the project into Colab, uses Colab's preinstalled scientific stack, runs Assignment 1 on CPU.

In [1]:
import os
import platform
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

Python: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content


## Get the project

The cell clones the repository the first time and pulls new commits on later runs.

In [2]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/MinhTrinhh/deep-learning-landing-page.git"
REPOSITORY_DIR = Path("/content/deep-learning-landing-page")

if (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

print("Repository ready at:", REPOSITORY_DIR)

Repository ready at: /content/deep-learning-landing-page


In [3]:
from importlib.util import find_spec

colab_extras = {
    "thop": "thop==0.1.1.post2209072238",
    "umap": "umap-learn",
}
missing_packages = [
    package_name
    for import_name, package_name in colab_extras.items()
    if find_spec(import_name) is None
]

if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", *missing_packages],
        check=True,
    )
else:
    print("All Colab extras are already installed.")

In [4]:
from importlib.metadata import version

import matplotlib
import numpy as np
import scipy
import seaborn
import sklearn
import thop
import torch
import torchvision
import umap

versions = {
    "PyTorch": torch.__version__,
    "Torchvision": torchvision.__version__,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Scikit-learn": sklearn.__version__,
    "Matplotlib": matplotlib.__version__,
    "Seaborn": seaborn.__version__,
    "UMAP-learn": version("umap-learn"),
    "THOP": version("thop"),
}
for package, package_version in versions.items():
    print(f"{package}: {package_version}")

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"Assigned GPU: {torch.cuda.get_device_name(0)}")
print("Experiment device: CPU (the project does not move models or tensors to CUDA)")

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
NumPy: 2.1.3
SciPy: 1.16.3
Scikit-learn: 1.6.1
Matplotlib: 3.10.0
Seaborn: 0.13.2
UMAP-learn: 0.5.12
THOP: 0.1.1-2209072238
CUDA available: False
Experiment device: CPU (the project does not move models or tensors to CUDA)


## Configure and run an experiment

Choose one model family and whether to use augmentation or regenerate the EDA outputs. The project code currently trains on CPU.

In [5]:
MODEL = "cnn"             # "linear", "mlp", "cnn", or "all"
USE_AUGMENTATION = True       # False adds --no-aug
RUN_EDA = False               # True regenerates EDA before training

assert MODEL in {"linear", "mlp", "cnn", "all"}
print({"model": MODEL, "augmentation": USE_AUGMENTATION, "eda": RUN_EDA})

{'model': 'cnn', 'augmentation': True, 'eda': False}


In [6]:
ASSIGNMENT_DIR = REPOSITORY_DIR / "Assignment1"
CODE_DIR = ASSIGNMENT_DIR / "code"
LOG_PATH = ASSIGNMENT_DIR / "outputs" / "log.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

command = [sys.executable, "-u", "main.py", "--model", MODEL]
command.append("--aug" if USE_AUGMENTATION else "--no-aug")
if RUN_EDA:
    command.append("--eda")

experiment_environment = os.environ.copy()
experiment_environment["PYTHONUNBUFFERED"] = "1"

run_message = "Running: " + " ".join(command)
print(run_message, flush=True)
print("Writing complete run output to:", LOG_PATH, flush=True)

with LOG_PATH.open("w", encoding="utf-8", buffering=1) as log_file:
    log_file.write(run_message + "\n")
    with subprocess.Popen(
        command,
        cwd=CODE_DIR,
        env=experiment_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        assert process.stdout is not None
        for output_line in process.stdout:
            print(output_line, end="", flush=True)
            log_file.write(output_line)
        return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

Running: /usr/bin/python3 -u main.py --model cnn --aug
Writing complete run output to: /content/deep-learning-landing-page/Assignment1/outputs/log.txt
I0000 00:00:1791264881.689963    4445 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791264885.282500    4445 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
usage: main.py [-h] [--eda] [--model {linear,mlp,all}] [--aug | --no-aug]
main.py: error: argument --model: invalid choice: 'cnn' (choose from 'linear', 'mlp', 'all')


CalledProcessError: Command '['/usr/bin/python3', '-u', 'main.py', '--model', 'cnn', '--aug']' returned non-zero exit status 2.

## Inspect and download artifacts

The final cell creates one ZIP archive containing `checkpoints/` and `outputs/`, including the complete `outputs/log.txt` training and evaluation log. Download it from the Colab Contents panel in VS Code.

In [ ]:
ASSIGNMENT_DIR = REPOSITORY_DIR / "Assignment1"
artifact_roots = [ASSIGNMENT_DIR / "checkpoints", ASSIGNMENT_DIR / "outputs"]

for artifact_root in artifact_roots:
    print(f"\n{artifact_root.relative_to(REPOSITORY_DIR)}/")
    if artifact_root.exists():
        for artifact in sorted(path for path in artifact_root.rglob("*") if path.is_file()):
            print(" -", artifact.relative_to(artifact_root))
    else:
        print(" - No artifacts generated yet")

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/content/assignment1-artifacts.zip")
included_files = 0

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for directory_name in ("checkpoints", "outputs"):
        directory = ASSIGNMENT_DIR / directory_name
        if not directory.exists():
            print(f"Skipped missing directory: {directory}")
            continue

        for source_path in sorted(path for path in directory.rglob("*") if path.is_file()):
            archive_name = Path("Assignment1") / source_path.relative_to(ASSIGNMENT_DIR)
            archive.write(source_path, archive_name)
            included_files += 1

if included_files == 0:
    archive_path.unlink(missing_ok=True)
    raise FileNotFoundError("No checkpoint or output files were found. Run an experiment first.")

archive_size_mb = archive_path.stat().st_size / (1024 ** 2)
print(f"Created {archive_path} with {included_files} files ({archive_size_mb:.2f} MB)")
print("Download it from the VS Code Colab Contents view.")